In [2]:
!pip install xarray cfgrib netCDF4 pandas requests

  Using cached cfgrib-0.9.15.1-py3-none-any.whl.metadata (56 kB)
  Using cached netcdf4-1.7.4-cp311-abi3-win_amd64.whl.metadata (2.1 kB)
  Using cached findlibs-0.1.3-py3-none-any.whl.metadata (4.3 kB)
Using cached cfgrib-0.9.15.1-py3-none-any.whl (49 kB)
Using cached netcdf4-1.7.4-cp311-abi3-win_amd64.whl (21.3 MB)
   ---------------------------------------- 0.0/7.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.6 MB 3.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.0/7.6 MB 3.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.0/7.6 MB 3.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.0/7.6 MB 3.2 MB/s eta 0:00:03
   ----------- ---------------------------- 2.1/7.6 MB 1.7 MB/s eta 0:00:04
   ----------- ---------------------------- 2.1/7.6 MB 1.7 MB/s eta 0:00:04
   ---------------- ----------------------- 3.1/7.6 MB 1.9 MB/s eta 0:00:03
   ---------

In [1]:
import os
import requests
import xarray as xr
import pandas as pd
from pathlib import Path
from urllib.parse import urlencode

In [2]:
# ==============================
# USER CONFIGURATION
# ==============================

FEATURES = [
    "temperature",
    "pressure"
]

PARAMETERS = {
    "temperature": "2 m above ground",
    "pressure": "Mean Sea Level Pressure"
}

LATITUDE = (5, 35)
LONGITUDE = (65, 100)

# GFS settings
DATE = "20260910"
CYCLE = "00"
FORECAST_HOUR = "000"

# Output
OUTPUT_DIR = Path("gfs_output")

OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
# ==============================
# GFS FEATURE MAPPING
# ==============================

FEATURE_MAP = {
    "temperature": {
        "variable": "TMP",
        "level": "lev_2_m_above_ground"
    },

    "pressure": {
        "variable": "PRMSL",
        "level": "lev_mean_sea_level"
    }
}

In [4]:
# ==============================
# BUILD GFS REQUEST URL
# ==============================

def build_gfs_url():
    
    variables = set()
    levels = set()

    for feature in FEATURES:
        
        if feature not in FEATURE_MAP:
            raise ValueError(f"Unsupported feature: {feature}")
        
        variables.add(
            FEATURE_MAP[feature]["variable"]
        )
        
        levels.add(
            FEATURE_MAP[feature]["level"]
        )

    params = {
        "file": f"gfs.t{CYCLE}z.pgrb2.0p25.f{FORECAST_HOUR}",

        "all_lev": "off",
        "all_var": "off",

        "subregion": "",

        "leftlon": LONGITUDE[0],
        "rightlon": LONGITUDE[1],

        "toplat": LATITUDE[1],
        "bottomlat": LATITUDE[0],

        "dir": f"/gfs.{DATE}/{CYCLE}/atmos"
    }

    # Add variables
    for var in variables:
        params[f"var_{var}"] = "on"

    # Add levels
    for level in levels:
        params[level] = "on"

    base_url = "https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_0p25.pl"

    return base_url + "?" + urlencode(params)

In [5]:
url = build_gfs_url()

print(url)

https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_0p25.pl?file=gfs.t00z.pgrb2.0p25.f000&all_lev=off&all_var=off&subregion=&leftlon=65&rightlon=100&toplat=35&bottomlat=5&dir=%2Fgfs.20260910%2F00%2Fatmos&var_PRMSL=on&var_TMP=on&lev_mean_sea_level=on&lev_2_m_above_ground=on


In [6]:
# ==============================
# DOWNLOAD GFS DATA
# ==============================

def download_gfs(url):
    
    output_file = OUTPUT_DIR / "gfs_data.grib2"

    print("Downloading GFS data...")
    print(url)

    response = requests.get(url, timeout=120)

    response.raise_for_status()

    with open(output_file, "wb") as f:
        f.write(response.content)

    print("Download completed.")
    print(f"Saved at: {output_file}")

    return output_file

In [7]:
grib_file = download_gfs(url)

https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_0p25.pl?file=gfs.t00z.pgrb2.0p25.f000&all_lev=off&all_var=off&subregion=&leftlon=65&rightlon=100&toplat=35&bottomlat=5&dir=%2Fgfs.20260910%2F00%2Fatmos&var_PRMSL=on&var_TMP=on&lev_mean_sea_level=on&lev_2_m_above_ground=on
Download completed.
Saved at: gfs_output\gfs_data.grib2


In [8]:
import dask
print(dask.__version__)

2026.8.0


In [9]:
!pip install --upgrade dask

In [10]:
import dask
print(dask.__version__)

2026.8.0


In [11]:
import os
import requests
import xarray as xr
import pandas as pd
from pathlib import Path
from urllib.parse import urlencode

In [12]:
print(xr.__version__)

2025.10.1


In [13]:
temperature = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "2t"
        }
    }
)

print("Temperature dataset successfully opened!")
print(temperature.data_vars)
print(temperature.coords)

Ignoring index file 'gfs_output\\gfs_data.grib2.47d85.idx' older than GRIB file


Temperature dataset successfully opened!
Data variables:
    t2m      (latitude, longitude) float32 68kB ...
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    time               datetime64[ns] 8B ...
    step               timedelta64[ns] 8B ...
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B ...


In [14]:
pressure = xr.open_dataset(
    grib_file,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {
            "shortName": "prmsl"
        }
    }
)

print("Pressure dataset successfully opened!")
print(pressure.data_vars)
print(pressure.coords)

Ignoring index file 'gfs_output\\gfs_data.grib2.47d85.idx' older than GRIB file


Pressure dataset successfully opened!
Data variables:
    prmsl    (latitude, longitude) float32 68kB ...
Coordinates:
  * latitude    (latitude) float64 968B 5.0 5.25 5.5 5.75 ... 34.5 34.75 35.0
  * longitude   (longitude) float64 1kB 65.0 65.25 65.5 ... 99.5 99.75 100.0
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    meanSea     float64 8B ...
    valid_time  datetime64[ns] 8B ...


In [15]:
# Temperature: Kelvin → Celsius
temperature_c = temperature["t2m"] - 273.15

# Pressure: Pa → hPa
pressure_hpa = pressure["prmsl"] / 100

print("Temperature range:")
print(float(temperature_c.min()), "to", float(temperature_c.max()), "°C")

print("\nPressure range:")
print(float(pressure_hpa.min()), "to", float(pressure_hpa.max()), "hPa")

Temperature range:
-8.4744873046875 to 31.925506591796875 °C

Pressure range:
1002.6375122070312 to 1029.3695068359375 hPa


In [16]:
final_dataset = xr.Dataset(
    {
        "temperature_2m_C": temperature_c,
        "pressure_msl_hPa": pressure_hpa
    }
)

print("Final dataset created!")
print(final_dataset)

Final dataset created!
<xarray.Dataset> Size: 139kB
Dimensions:            (latitude: 121, longitude: 141)
Coordinates:
  * latitude           (latitude) float64 968B 5.0 5.25 5.5 ... 34.5 34.75 35.0
  * longitude          (longitude) float64 1kB 65.0 65.25 65.5 ... 99.75 100.0
    time               datetime64[ns] 8B 2026-09-10
    step               timedelta64[ns] 8B 00:00:00
    heightAboveGround  float64 8B ...
    valid_time         datetime64[ns] 8B 2026-09-10
    meanSea            float64 8B ...
Data variables:
    temperature_2m_C   (latitude, longitude) float32 68kB 25.33 25.63 ... 3.026
    pressure_msl_hPa   (latitude, longitude) float32 68kB 1.012e+03 ... 1.026...


In [17]:
df = final_dataset.to_dataframe().reset_index()

df = df[
    [
        "time",
        "latitude",
        "longitude",
        "temperature_2m_C",
        "pressure_msl_hPa"
    ]
]

csv_file = OUTPUT_DIR / "gfs_weather_data.csv"

df.to_csv(csv_file, index=False)

print("CSV saved successfully!")
print("File:", csv_file)

print("\nShape:", df.shape)
print("\nFirst 10 rows:")
print(df.head(10))

CSV saved successfully!
File: gfs_output\gfs_weather_data.csv

Shape: (17061, 5)

First 10 rows:
        time  latitude  longitude  temperature_2m_C  pressure_msl_hPa
0 2026-09-10       5.0      65.00         25.325531       1012.197510
1 2026-09-10       5.0      65.25         25.625519       1011.943542
2 2026-09-10       5.0      65.50         25.725525       1012.305481
3 2026-09-10       5.0      65.75         26.125519       1012.231506
4 2026-09-10       5.0      66.00         26.725525       1012.121460
5 2026-09-10       5.0      66.25         26.925507       1012.001465
6 2026-09-10       5.0      66.50         27.125519       1011.913513
7 2026-09-10       5.0      66.75         27.225525       1011.891479
8 2026-09-10       5.0      67.00         27.225525       1011.883545
9 2026-09-10       5.0      67.25         27.325531       1011.781494


In [18]:
netcdf_file = OUTPUT_DIR / "gfs_weather_data.nc"

final_dataset.to_netcdf(netcdf_file)

print("NetCDF saved successfully!")
print("File:", netcdf_file)

NetCDF saved successfully!
File: gfs_output\gfs_weather_data.nc


In [19]:
print("Dataset Shape:")
print(df.shape)

print("\nFirst 10 rows:")
display(df.head(10))

print("\nMissing Values:")
print(df.isna().sum())

print("\nStatistics:")
display(
    df[
        ["temperature_2m_C", "pressure_msl_hPa"]
    ].describe()
)

Dataset Shape:
(17061, 5)

First 10 rows:


,time,latitude,longitude,temperature_2m_C,pressure_msl_hPa
0,2026-09-10,5.0,65.00,25.325531,1012.197510
1,2026-09-10,5.0,65.25,25.625519,1011.943542
2,2026-09-10,5.0,65.50,25.725525,1012.305481
3,2026-09-10,5.0,65.75,26.125519,1012.231506
4,2026-09-10,5.0,66.00,26.725525,1012.121460
5,2026-09-10,5.0,66.25,26.925507,1012.001465
6,2026-09-10,5.0,66.50,27.125519,1011.913513
7,2026-09-10,5.0,66.75,27.225525,1011.891479
8,2026-09-10,5.0,67.00,27.225525,1011.883545
9,2026-09-10,5.0,67.25,27.325531,1011.781494



Missing Values:
time                0
latitude            0
longitude           0
temperature_2m_C    0
pressure_msl_hPa    0
dtype: int64

Statistics:


,temperature_2m_C,pressure_msl_hPa
count,17061.000000,17061.000000
mean,22.302671,1010.437134
std,8.630519,4.658672
min,-8.474487,1002.637512
25%,21.625519,1007.419556
50%,26.425507,1009.567505
75%,27.825531,1011.425476
max,31.925507,1029.369507
